In [ ]:
import xgboost
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from datetime import datetime
from src import experiment_tracking
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from time import perf_counter
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)

In [ ]:
threshold_inicial_value = 0.50

n_estimators_value = 1000
learning_rate_value = 0.05

max_depth_value = 4
min_child_weight_value = 3
gamma_value = 0.0

subsample_value = 0.80
colsample_bytree_value = 0.80

reg_alpha_value = 0.0
reg_lambda_value = 1.0

scale_pos_weight_value = 1.0

objective_value = "binary:logistic"
eval_metric_value = "aucpr"
tree_method_value = "hist"

early_stopping_rounds_value = 50

n_jobs_value = -1
random_state_value = 42

nome_experimento = (
    "xgboost_spw1_lr005_d6_"
    "child3_sub08_col08_"
    "earlystop50_threshold_050"
)

observacao = (
    "Primeira execução do XGBoost. "
    "Foram utilizadas as mesmas features e a mesma "
    "separação temporal 70/15/15 dos modelos anteriores. "
    "O treinamento utilizou early stopping orientado pela "
    "métrica aucpr. Não foi aplicada ponderação adicional "
    "à classe fraudulenta."
)

In [ ]:
df = pd.read_csv("../raw/fraudTrain.csv", parse_dates = ["trans_date_trans_time", "dob"], dtype = {"cc_num": "string", "trans_num": "string", "zip": "string"})

In [ ]:
# variaveis que serão utilizadas

variaveis_numericas = [
    "amt_log",
    "city_pop_log",
    "idade",
    "distancia_km",
    "hora_sin",
    "hora_cos",
    "dia_semana_sin",
    "dia_semana_cos",
    "fim_de_semana"
]

variaveis_categoricas = [
   "category",
   "gender",
   "state"
]

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

def avaliar_modelo(
    y_real,
    scores,
    threshold
) -> pd.DataFrame:
    """
    Avalia um modelo de classificação binária a partir dos scores
    atribuídos à classe positiva.

    Parâmetros
    ----------
    y_real:
        Classes reais. Espera-se 0 para transação legítima e
        1 para transação fraudulenta.

    scores:
        Probabilidade ou score atribuído à classe de fraude.

        Exemplo:
        modelo.predict_proba(X)[:, 1]

    threshold:
        Ponto de corte utilizado para transformar o score em classe.

        score >= threshold -> fraude
        score < threshold  -> legítima

    Retorno
    -------
    pd.DataFrame:
        DataFrame com uma linha contendo todas as métricas.
    """

    y_real = np.asarray(y_real).ravel()
    scores = np.asarray(scores).ravel()

    if len(y_real) != len(scores):
        raise ValueError(
            "y_real e scores precisam possuir a mesma quantidade "
            "de observações."
        )

    if not 0 <= threshold <= 1:
        raise ValueError(
            "O threshold deve estar entre 0 e 1."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Os scores possuem valores ausentes ou infinitos."
        )

    # Converte o score em uma decisão binária.
    y_previsto = (scores >= threshold).astype(int)

    # labels=[0, 1] garante uma matriz 2x2 mesmo quando alguma
    # classe não é prevista pelo modelo.
    matriz = confusion_matrix(
        y_real,
        y_previsto,
        labels=[0, 1]
    )

    verdadeiros_negativos, falsos_positivos, \
        falsos_negativos, verdadeiros_positivos = matriz.ravel()

    # ROC-AUC exige que existam as duas classes em y_real.
    if np.unique(y_real).size == 2:
        roc_auc = roc_auc_score(y_real, scores)
    else:
        roc_auc = np.nan

    resultado = {
        "threshold": float(threshold),

        # Métricas calculadas diretamente sobre os scores.
        "average_precision": average_precision_score(
            y_real,
            scores
        ),
        "roc_auc": roc_auc,

        # Métricas dependentes do threshold.
        "accuracy": accuracy_score(
            y_real,
            y_previsto
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_previsto
        ),
        "precision_fraude": precision_score(
            y_real,
            y_previsto,
            pos_label=1,
            zero_division=0
        ),
        "recall_fraude": recall_score(
            y_real,
            y_previsto,
            pos_label=1,
            zero_division=0
        ),
        "f1_fraude": f1_score(
            y_real,
            y_previsto,
            pos_label=1,
            zero_division=0
        ),

        # Valores da matriz de confusão.
        "verdadeiros_negativos": int(verdadeiros_negativos),
        "falsos_positivos": int(falsos_positivos),
        "falsos_negativos": int(falsos_negativos),
        "verdadeiros_positivos": int(verdadeiros_positivos),

        # Percentual de transações encaminhadas como fraude.
        "percentual_alertado": y_previsto.mean() * 100
    }

    return pd.DataFrame([resultado])

In [ ]:
# modelo_random_forest = RandomForestClassifier(
#     n_estimators=n_estimators_value,
#     criterion=criterion_value,
#     max_depth=max_depth_value,
#     min_samples_split=min_samples_split_value,
#     min_samples_leaf=min_samples_leaf_value,
#     max_features=max_features_value,
#     bootstrap=bootstrap_value,
#     class_weight=class_weight_value,
#     n_jobs=n_jobs_value,
#     random_state=random_state_value
# )

In [ ]:
# parametros_random_forest = {
#     "n_estimators":n_estimators_value,
#     "criterion":criterion_value,
#     "max_depth":max_depth_value,
#     "min_samples_split":min_samples_split_value,
#     "min_samples_leaf":min_samples_leaf_value,
#     "max_features":max_features_value,
#     "bootstrap":bootstrap_value,
#     "class_weight":class_weight_value,
#     "n_jobs":n_jobs_value,
#     "random_state":random_state_value,
#     "threshold":threshold_inicial_value
# }

# nome_experimento = (
#     "random_forest_none_"
#     "n200_d20_leaf5_threshold_050"
# )

# observacao = (
#     "Primeira execução da Random Forest. "
#     "Mesmas features e separação temporal utilizadas "
#     "na baseline de Regressão Logística. "
#     "Sem ponderação das classes."
# )

In [ ]:
# Removendo a coluna de índice
colunas_sem_nome = [coluna for coluna in df.columns if coluna.startswith("Unnamed")]

df = df.drop(columns=colunas_sem_nome)

In [ ]:
# Ordenando de maneira cronológica
df = (df.sort_values("trans_date_trans_time").reset_index(drop = True))

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"Início: {df['trans_date_trans_time'].min()}")
print(f"Fim: {df['trans_date_trans_time'].max()}")

In [ ]:

CAMINHO_RESULTADOS = Path(
    "../outputs/resultados_modelos_xgboost.csv"
)


def salvar_resultado_experimento(
    resultado,
    nome_modelo: str,
    nome_experimento: str,
    parametros: dict ,
    conjunto_features: str = "",
    observacoes: str = "",
    tempo_treinamento_segundos: float = "",
    caminho: Path = CAMINHO_RESULTADOS
) -> pd.DataFrame:
    """
    Salva uma execução do modelo em um arquivo CSV.

    Cada linha do arquivo representa um experimento.
    """

    caminho.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Aceita tanto um DataFrame de uma linha quanto um dicionário.
    if isinstance(resultado, pd.DataFrame):
        if len(resultado) != 1:
            raise ValueError(
                "O DataFrame de resultado deve possuir exatamente uma linha."
            )

        registro = resultado.iloc[0].to_dict()

    elif isinstance(resultado, dict):
        registro = resultado.copy()

    else:
        raise TypeError(
            "resultado deve ser um DataFrame de uma linha ou um dicionário."
        )

    metadados = {
        "data_execucao": datetime.now().isoformat(
            timespec="seconds"
        ),
        "nome_experimento": nome_experimento,
        "nome_modelo": nome_modelo,
        "conjunto_features": conjunto_features,
        "parametros": json.dumps(
            parametros or {},
            ensure_ascii=False,
            sort_keys=True
        ),
        "tempo_treinamento_segundos": (
            tempo_treinamento_segundos
        ),
        "observacoes": observacoes
    }

    registro_completo = {
        **metadados,
        **registro
    }

    nova_linha = pd.DataFrame(
        [registro_completo]
    )

    arquivo_ja_existe = caminho.exists()

    nova_linha.to_csv(
        caminho,
        mode="a",
        header=not arquivo_ja_existe,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Experimento salvo em: {caminho.resolve()}"
    )

    return nova_linha

In [ ]:
def calcular_distancia_km(
        lat_cliente: pd.Series,
        lon_cliente: pd.Series,
        lat_estabelecimento: pd.Series,
        lon_estabelecimento: pd.Series
) -> np.ndarray:
    """
    Calcula a distancia entre dois pontos geográficos utilizando a fórmula de Haversine.
    """

    raio_terra_km = 6371.0088

    lat1 = np.radians(lat_cliente.to_numpy())
    lon1 = np.radians(lon_cliente.to_numpy())
    lat2 = np.radians(lat_estabelecimento.to_numpy())
    lon2 = np.radians(lon_estabelecimento.to_numpy())

    delta_lat = lat2 - lat1
    delta_lon  = lon2 - lon1

    a = (np.sin(delta_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2 )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return raio_terra_km * c

In [ ]:
def criar_features(dados: pd.DataFrame) -> pd.DataFrame:
    """
    Cria as variáveis utilizadas pela Regressão Logística.
    
    Todas as variáveis são construídas usando apenas informações da própria transação.
    """

    features = pd.DataFrame(index = dados.index)

    data_transacao = dados["trans_date_trans_time"]
    data_nascimento = dados["dob"]

    # Valor da transação em escala logarítmica
    features["amt_log"] = np.log1p(dados["amt"])

    # População também apresenta assimetria
    features["city_pop_log"] = np.log1p(dados["city_pop"])

    # Idade exata na data da transação
    aniversario_ainda_nao_ocorreu = (data_transacao.dt.month < data_nascimento.dt.month) | ((data_transacao.dt.month == data_nascimento.dt.month) & (data_transacao.dt.day < data_nascimento.dt.day))

    features["idade"] = (
        data_transacao.dt.year - data_nascimento.dt.year - aniversario_ainda_nao_ocorreu.astype(int)
    )

    # Distância aproximada entre cliente e estabelecimento
    features["distancia_km"] = calcular_distancia_km(
        dados["lat"],
        dados["long"],
        dados["merch_lat"],
        dados["merch_long"]
    )

    # Variáveis temporais cíclicas
    hora = data_transacao.dt.hour

    features["hora_sin"] = np.sin(
        2 * np.pi * hora / 24
    )

    features["hora_cos"] = np.cos(
        2 * np.pi * hora / 24
    )

    dia_semana = data_transacao.dt.dayofweek

    features["dia_semana_sin"] = np.sin(
        2 * np.pi * dia_semana / 7
    )
    
    features["dia_semana_cos"] = np.cos(
        2 * np.pi * dia_semana / 7
    )

    features["fim_de_semana"] = (
        dia_semana >= 5
    ).astype(int)

    # variáveis categóricas
    features["category"] = dados["category"].astype("string")
    features["gender"] = dados["gender"].astype("string")
    features["state"] = dados["state"].astype("string")

    return features

In [ ]:
# Separação temporal
# 70% dos dados mais antigos -> treino
# 15% seguinte -> validação
# 15% mais recente -> teste

limite_treino = int(len(df) * 0.7)
limite_validacao = int(len(df) * 0.85)

df_treino = df.iloc[:limite_treino].copy()

df_validacao = df.iloc[limite_treino:limite_validacao].copy()

df_teste = df.iloc[limite_validacao:].copy()

In [ ]:
# criando as features
X_treino = criar_features(df_treino)
X_validacao = criar_features(df_validacao)
X_teste = criar_features(df_teste)

y_treino = df_treino["is_fraud"].astype(int)
y_validacao = df_validacao["is_fraud"].astype(int)
y_teste = df_teste["is_fraud"].astype(int)

In [ ]:
import numpy as np

quantidade_legitimas_treino = int(
    np.sum(np.asarray(y_treino) == 0)
)

quantidade_fraudes_treino = int(
    np.sum(np.asarray(y_treino) == 1)
)

razao_desbalanceamento = (
    quantidade_legitimas_treino
    / quantidade_fraudes_treino
)

print(
    "Transações legítimas no treino:",
    quantidade_legitimas_treino
)

print(
    "Fraudes no treino:",
    quantidade_fraudes_treino
)

print(
    "Razão entre legítimas e fraudes:",
    f"{razao_desbalanceamento:.2f}"
)

In [ ]:
def resumo_particao(nome: str, dados: pd.DataFrame, target: pd.Series) -> dict:
    return {
        "particao": nome,
        "inicio": dados["trans_date_trans_time"].min(), 
        "fim": dados["trans_date_trans_time"].max(),
        "transacoes": len(dados),
        "fraudes": int(target.sum()),
        "taxa_fraude_percentual": target.mean() * 100
    }

resumo_particoes = pd.DataFrame([
    resumo_particao("Treino", df_treino, y_treino),
    resumo_particao("Validação", df_validacao, y_validacao),
    resumo_particao("Teste", df_teste, y_teste)
])

In [ ]:
# experimento = experiment_tracking.iniciar_experimento(
#     nome_experimento
# )

# parametros_experimento = {
#     "modelo": "RandomForestClassifier",
#     "n_estimators": n_estimators_value,
#     "criterion": criterion_value,
#     "max_depth": max_depth_value,
#     "min_samples_split": min_samples_split_value,
#     "min_samples_leaf": min_samples_leaf_value,
#     "max_features": max_features_value,
#     "bootstrap": bootstrap_value,
#     "class_weight": class_weight_value,
#     "n_jobs": -1,
#     "threshold": threshold_inicial_value,
#     "random_state": random_state_value,
#     "split": "temporal_70_15_15",
#     "features": [
#         "amt_log",
#         "city_pop_log",
#         "idade",
#         "distancia_km",
#         "hora_sin",
#         "hora_cos",
#         "dia_semana_sin",
#         "dia_semana_cos",
#         "fim_de_semana",
#         "category",
#         "gender",
#         "state"
#     ]
# }

# experiment_tracking.salvar_json(
#     parametros_experimento,
#     experimento["pasta_run"] / "parametros.json"
# )

In [ ]:
pipeline_numerico_rf = Pipeline (
    steps = [
        (
            "imputacao",
            SimpleImputer(strategy = "median")
        )
    ]
)

pipeline_categorico_rf = Pipeline(
    steps = [
        (
            "imputacao",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "one_hot",
            OneHotEncoder(
                handle_unknown = "ignore"
            )
        )
    ]
)

In [ ]:
preprocessador_rf = ColumnTransformer(
    transformers = [
        (
        "numericas",
        pipeline_numerico_rf,
        variaveis_numericas
    ),
    (
        "categoricas",
        pipeline_categorico_rf,
        variaveis_categoricas
    )
    ],
    remainder="drop"
)

In [ ]:
preprocessador_xgb = clone(
    preprocessador_rf
)

In [ ]:
X_treino_xgb = (
    preprocessador_xgb
    .fit_transform(X_treino)
)

In [ ]:
X_validacao_xgb = (
    preprocessador_xgb
    .transform(X_validacao)
)

In [ ]:
print(
    "Treino transformado:",
    X_treino_xgb.shape
)

print(
    "Validação transformada:",
    X_validacao_xgb.shape
)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
modelo_xgboost = XGBClassifier(
    n_estimators=n_estimators_value,
    learning_rate=learning_rate_value,

    max_depth=max_depth_value,
    min_child_weight=min_child_weight_value,
    gamma=gamma_value,

    subsample=subsample_value,
    colsample_bytree=colsample_bytree_value,

    reg_alpha=reg_alpha_value,
    reg_lambda=reg_lambda_value,

    scale_pos_weight=scale_pos_weight_value,

    objective=objective_value,
    eval_metric=eval_metric_value,
    tree_method=tree_method_value,

    early_stopping_rounds=(
        early_stopping_rounds_value
    ),

    n_jobs=n_jobs_value,
    random_state=random_state_value
)

In [ ]:
inicio_treinamento = perf_counter()

modelo_xgboost.fit(
    X_treino_xgb,
    y_treino,
    eval_set=[
        (
            X_validacao_xgb,
            y_validacao
        )
    ],
    verbose=50
)

tempo_treinamento = (
    perf_counter()
    - inicio_treinamento
)

print(
    f"Tempo de treinamento: "
    f"{tempo_treinamento:.2f} segundos"
)

In [ ]:
import platform
import sys
import xgboost as xgb

print("Arquitetura:", platform.machine())
print("Python:", sys.executable)
print("Python version:", sys.version)
print("XGBoost:", xgb.__version__)
print("Build:", xgb.build_info())

In [ ]:
print(
    "Melhor iteração:",
    modelo_xgboost.best_iteration
)

print(
    "Melhor score de validação:",
    modelo_xgboost.best_score
)

In [ ]:
quantidade_arvores_utilizadas = (
    modelo_xgboost.best_iteration + 1
)

print(
    "Quantidade efetiva de árvores:",
    quantidade_arvores_utilizadas
)

In [ ]:
scores_validacao_xgb = (
    modelo_xgboost
    .predict_proba(X_validacao_xgb)[:, 1]
)

In [ ]:
import pandas as pd

pd.Series(
    scores_validacao_xgb
).describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
        0.995,
        0.999
    ]
)

In [ ]:
resultado_xgboost_threshold_05 = avaliar_modelo(
    y_real=y_validacao,
    scores=scores_validacao_xgb,
    threshold=threshold_inicial_value
)

resultado_xgboost_threshold_05.T

In [ ]:
parametros_experimento = {
    "n_estimators_maximo": (
        n_estimators_value
    ),
    "best_iteration": int(
        modelo_xgboost.best_iteration
    ),
    "quantidade_arvores_utilizadas": int(
        quantidade_arvores_utilizadas
    ),
    "learning_rate": learning_rate_value,
    "max_depth": max_depth_value,
    "min_child_weight": (
        min_child_weight_value
    ),
    "gamma": gamma_value,
    "subsample": subsample_value,
    "colsample_bytree": (
        colsample_bytree_value
    ),
    "reg_alpha": reg_alpha_value,
    "reg_lambda": reg_lambda_value,
    "scale_pos_weight": (
        scale_pos_weight_value
    ),
    "objective": objective_value,
    "eval_metric_treinamento": (
        eval_metric_value
    ),
    "tree_method": tree_method_value,
    "early_stopping_rounds": (
        early_stopping_rounds_value
    ),
    "threshold": threshold_inicial_value,
    "random_state": random_state_value
}

In [ ]:
salvar_resultado_experimento(
    resultado=resultado_xgboost_threshold_05,
    nome_modelo="XGBoost",
    nome_experimento=nome_experimento,
    parametros=parametros_experimento,
    conjunto_features="features_basicas_v1",
    tempo_treinamento_segundos=tempo_treinamento,
    observacoes=observacao
)

In [ ]:
comparacao_modelos = pd.DataFrame({
    "modelo": [
        "Regressão Logística",
        "Random Forest",
        "XGBoost"
    ],
    "average_precision": [
        0.401716,
        0.900570,
        resultado_xgboost_threshold_05.loc[
            0,
            "average_precision"
        ]
    ],
    "roc_auc": [
        0.890614,
        0.991924,
        resultado_xgboost_threshold_05.loc[
            0,
            "roc_auc"
        ]
    ],
    "precision_fraude": [
        0.671795,
        0.983070,
        resultado_xgboost_threshold_05.loc[
            0,
            "precision_fraude"
        ]
    ],
    "recall_fraude": [
        0.104633,
        0.695687,
        resultado_xgboost_threshold_05.loc[
            0,
            "recall_fraude"
        ]
    ],
    "f1_fraude": [
        0.181064,
        0.814780,
        resultado_xgboost_threshold_05.loc[
            0,
            "f1_fraude"
        ]
    ]
})

comparacao_modelos

In [ ]:
PASTA_FIGURAS_XGB = Path(
    "../outputs/figures/xgboost"
)

PASTA_TABELAS_XGB = Path(
    "../outputs/tables/xgboost"
)

PASTA_FIGURAS_XGB.mkdir(
    parents=True,
    exist_ok=True
)

PASTA_TABELAS_XGB.mkdir(
    parents=True,
    exist_ok=True
)

def salvar_e_exibir_figura_xgb(
    fig,
    nome_arquivo: str
) -> None:
    """
    Salva uma figura do XGBoost em PNG com 300 DPI.
    """

    fig.tight_layout()

    caminho = (
        PASTA_FIGURAS_XGB
        / f"{nome_arquivo}.png"
    )

    fig.savefig(
        caminho,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"Figura salva em: {caminho.resolve()}"
    )

In [ ]:
scores_validacao_xgb = (
    modelo_xgboost
    .predict_proba(X_validacao_xgb)[:, 1]
)

In [ ]:
threshold_xgb = threshold_inicial_value

y_real_xgb = np.asarray(
    y_validacao
).ravel()

scores_validacao_xgb = np.asarray(
    scores_validacao_xgb
).ravel()

y_previsto_xgb = (
    scores_validacao_xgb >= threshold_xgb
).astype(int)

In [ ]:
print(
    "Quantidade de observações:",
    len(y_real_xgb)
)

print(
    "Quantidade alertada:",
    y_previsto_xgb.sum()
)

print(
    "Percentual alertado:",
    f"{y_previsto_xgb.mean() * 100:.4f}%"
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 6)
)

ConfusionMatrixDisplay.from_predictions(
    y_true=y_real_xgb,
    y_pred=y_previsto_xgb,
    display_labels=[
        "Legítima",
        "Fraude"
    ],
    values_format="d",
    ax=ax
)

ax.set_title(
    "Matriz de confusão — XGBoost\n"
    f"Threshold = {threshold_xgb:.2f}"
)

ax.set_xlabel(
    "Classe prevista"
)

ax.set_ylabel(
    "Classe real"
)

salvar_e_exibir_figura_xgb(
    fig,
    "01_matriz_confusao_xgboost"
)

In [ ]:
taxa_falsos_positivos_xgb, \
taxa_verdadeiros_positivos_xgb, \
thresholds_roc_xgb = roc_curve(
    y_real_xgb,
    scores_validacao_xgb
)

roc_auc_xgb = roc_auc_score(
    y_real_xgb,
    scores_validacao_xgb
)

In [ ]:
taxa_falsos_positivos_xgb, \
taxa_verdadeiros_positivos_xgb, \
thresholds_roc_xgb = roc_curve(
    y_real_xgb,
    scores_validacao_xgb
)

roc_auc_xgb = roc_auc_score(
    y_real_xgb,
    scores_validacao_xgb
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    taxa_falsos_positivos_xgb,
    taxa_verdadeiros_positivos_xgb,
    label=(
        "XGBoost "
        f"(ROC-AUC = {roc_auc_xgb:.4f})"
    )
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Classificador aleatório"
)

ax.set_title(
    "Curva ROC — XGBoost"
)

ax.set_xlabel(
    "Taxa de falsos positivos"
)

ax.set_ylabel(
    "Taxa de verdadeiros positivos"
)

ax.legend()

salvar_e_exibir_figura_xgb(
    fig,
    "02_curva_roc_xgboost"
)

In [ ]:
precision_xgb, recall_xgb, thresholds_pr_xgb = (
    precision_recall_curve(
        y_real_xgb,
        scores_validacao_xgb
    )
)

average_precision_xgb = (
    average_precision_score(
        y_real_xgb,
        scores_validacao_xgb
    )
)

prevalencia_fraude = y_real_xgb.mean()

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    recall_xgb,
    precision_xgb,
    label=(
        "XGBoost "
        f"(AP = {average_precision_xgb:.4f})"
    )
)

ax.axhline(
    y=prevalencia_fraude,
    linestyle="--",
    label=(
        "Baseline pela prevalência "
        f"({prevalencia_fraude:.4%})"
    )
)

ax.set_title(
    "Curva Precision-Recall — XGBoost"
)

ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1.05
)

ax.legend()

salvar_e_exibir_figura_xgb(
    fig,
    "03_curva_precision_recall_xgboost"
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    thresholds_pr_xgb,
    precision_xgb[:-1],
    label="Precision"
)

ax.plot(
    thresholds_pr_xgb,
    recall_xgb[:-1],
    label="Recall"
)

ax.axvline(
    x=threshold_xgb,
    linestyle="--",
    label=(
        "Threshold analisado "
        f"({threshold_xgb:.2f})"
    )
)

ax.set_title(
    "Precision e Recall por threshold — XGBoost"
)

ax.set_xlabel(
    "Threshold"
)

ax.set_ylabel(
    "Valor da métrica"
)

ax.set_xlim(
    0,
    1
)

ax.set_ylim(
    0,
    1.05
)

ax.legend()

salvar_e_exibir_figura_xgb(
    fig,
    "04_precision_recall_por_threshold_xgboost"
)

In [ ]:
scores_legitimas_xgb = (
    scores_validacao_xgb[
        y_real_xgb == 0
    ]
)

scores_fraudes_xgb = (
    scores_validacao_xgb[
        y_real_xgb == 1
    ]
)

In [ ]:
bins_scores = np.linspace(
    0,
    1,
    51
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.hist(
    scores_legitimas_xgb,
    bins=bins_scores,
    density=True,
    alpha=0.6,
    label="Transações legítimas"
)

ax.hist(
    scores_fraudes_xgb,
    bins=bins_scores,
    density=True,
    alpha=0.6,
    label="Transações fraudulentas"
)

ax.axvline(
    x=threshold_xgb,
    linestyle="--",
    label=(
        "Threshold analisado "
        f"({threshold_xgb:.2f})"
    )
)

ax.set_yscale(
    "log"
)

ax.set_title(
    "Distribuição dos scores — XGBoost"
)

ax.set_xlabel(
    "Score estimado de fraude"
)

ax.set_ylabel(
    "Densidade em escala logarítmica"
)

ax.legend()

salvar_e_exibir_figura_xgb(
    fig,
    "05_distribuicao_scores_xgboost"
)

In [ ]:
nomes_features_xgb = (
    preprocessador_xgb
    .get_feature_names_out()
)

importancias_xgb = (
    modelo_xgboost
    .feature_importances_
)

In [ ]:
if len(nomes_features_xgb) != len(importancias_xgb):
    raise ValueError(
        "A quantidade de nomes de features não corresponde "
        "à quantidade de importâncias do XGBoost."
    )

tabela_importancias_xgb = pd.DataFrame({
    "variavel": nomes_features_xgb,
    "importancia": importancias_xgb
})

tabela_importancias_xgb["variavel"] = (
    tabela_importancias_xgb["variavel"]
    .str.replace(
        r"^[^_]+__",
        "",
        regex=True
    )
)

tabela_importancias_xgb = (
    tabela_importancias_xgb
    .sort_values(
        "importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

tabela_importancias_xgb.head(20)

In [ ]:
top_importancias_xgb = (
    tabela_importancias_xgb
    .head(20)
    .sort_values(
        "importancia",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

ax.barh(
    top_importancias_xgb["variavel"],
    top_importancias_xgb["importancia"]
)

ax.set_title(
    "Variáveis com maior importância — XGBoost"
)

ax.set_xlabel(
    "Importância calculada pelo XGBoost"
)

ax.set_ylabel(
    "Variável"
)

salvar_e_exibir_figura_xgb(
    fig,
    "06_importancia_variaveis_xgboost"
)

In [ ]:
historico_xgb = (
    modelo_xgboost
    .evals_result()
)

historico_xgb

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

for nome_conjunto, metricas in historico_xgb.items():

    if "aucpr" in metricas:
        nome_metrica = "aucpr"
    else:
        nome_metrica = next(
            iter(metricas)
        )

    valores = metricas[
        nome_metrica
    ]

    ax.plot(
        range(len(valores)),
        valores,
        label=nome_conjunto
    )

if hasattr(
    modelo_xgboost,
    "best_iteration"
):
    ax.axvline(
        x=modelo_xgboost.best_iteration,
        linestyle="--",
        label=(
            "Melhor iteração "
            f"({modelo_xgboost.best_iteration})"
        )
    )

ax.set_title(
    "Evolução da métrica durante o treinamento — XGBoost"
)

ax.set_xlabel(
    "Iteração de boosting"
)

ax.set_ylabel(
    "AUC-PR"
)

ax.legend()

salvar_e_exibir_figura_xgb(
    fig,
    "07_evolucao_treinamento_xgboost"
)

In [ ]:
print(
    "Melhor iteração:",
    modelo_xgboost.best_iteration
)

print(
    "Melhor score:",
    modelo_xgboost.best_score
)

print(
    "Total de árvores treinadas:",
    modelo_xgboost
    .get_booster()
    .num_boosted_rounds()
)